In [ ]:
from two_spring_block import CoupledSpringBlock
from matplotlib import pyplot as plt
import numpy as np
from joblib import Parallel, delayed
from tqdm import tqdm

In [ ]:
def get_single_phase(tq, t: np.ndarray):
    if tq < min(t) or tq > max(t):
        return np.NaN

    dt = (t - tq)

    if np.min(np.abs(dt)) == 0:
        index = np.argmin(np.abs(dt))
        t1 = 0
        if index == 0:
            T = dt[index + 1]
        elif index == len(dt) - 1:
            T = dt[index - 1]
        else:
            T = np.mean(dt[[index - 1, index + 1]])

    else:
        dt_pos = dt[dt >= 0]
        dt_neg = dt[dt < 0]

        t1 = min(-dt_neg)
        t2 = min(dt_pos)

        T = t1 + t2

    phase = 2 * np.pi * t1 / T

    return phase

In [ ]:
run_time = 1200*3.15e7
model = CoupledSpringBlock(
    sigma=np.array([10.0,9.5]), 
    k1=-1,
    k2=0.05,
    k3=0.05,
    Y=[-2,0,0,0],
    dc=0.0001,
)
model.simulate(run_time)


In [ ]:
fig, ax = plt.subplots(dpi=200, figsize=(6,1.25))
plt.rcParams['font.size'] = 8
ax.spines['top'].set_linewidth(0.5)
ax.spines['right'].set_linewidth(0.5)
ax.spines['bottom'].set_linewidth(0.5)
ax.spines['left'].set_linewidth(0.5)

model.plot_stress(ax=ax)
ax.set(
    xticks=np.arange(0, run_time, 100*3.15e7),
    xticklabels = [f'{int(_/3.15e7)}' for _ in np.arange(0,run_time, 100*3.15e7)],
    xlabel='Time (years)',
);

Note: This step takes around 20 minutes on a laptop.

In [ ]:
sigma_2 = np.linspace(6,10,300)
number_of_random_times = 1000
run_time = 2000*3.15e7

def simulate_model(i_sigma2):
    model = CoupledSpringBlock(
        sigma=np.array([10.0,i_sigma2]), 
        Y=[np.random.uniform(-10,0),0,np.random.uniform(-10,0),0],
        k1=-1,
        k2=0.1,
        k3=0.1,
    )
    model.simulate(run_time)
    random_times = np.random.uniform(0.8*run_time,run_time,number_of_random_times)
    Y1 = np.interp(random_times,model.time,model.slip_deficit[0,:])
    Y2 = np.interp(random_times,model.time,model.slip_deficit[1,:])
    
    tA, tB = model.get_events()
    phases = [get_single_phase(tq, tA) for tq in tB]
    
    return Y1 - Y2, i_sigma2, phases

results = Parallel(n_jobs=-1)(delayed(simulate_model)(i_sigma2) for i_sigma2 in tqdm(sigma_2))

delta_Y, sigma_2_list, phases_list = zip(*results)

In [ ]:
fig, ax = plt.subplots()
for i_sigma2, i_phases in zip(sigma_2_list, phases_list):
    ax.scatter(i_sigma2*np.ones_like(i_phases)/10,i_phases, s=5, c='k', alpha=0.02, linewidth=0)
ax.set(
    xlabel=r'Symmetry, $\sigma_2$',
    ylabel=r'Phase of A in B',
)

Making an iterate map of the phase difference between the two blocks.

We can make this by first getting the times of the events and then computing the phase of block A in the occurence of B. 

Next we can plot the phase 

In [ ]:
number_of_simulations = 10

run_time = 3000 * 3.15e7

def run_simulation(i):
    model = CoupledSpringBlock(
        sigma=np.array([10.0, 9.0]),
        k1=-1,
        k2=0.2,
        k3=0.2,
        Y=[np.random.uniform(-10, 0), 0, np.random.uniform(-10, 0), 0],
    )
    model.simulate(run_time)
        
    tA, tB = model.get_events()
    phases = [get_single_phase(tq, tA) for tq in tB]
    
    return model.time, model.slip_deficit[0, :] - model.slip_deficit[1, :], phases

results = Parallel(n_jobs=-1)(delayed(run_simulation)(i) for i in range(number_of_simulations))

In [ ]:
fig2, ax2 = plt.subplots(dpi=200, figsize=(3,2))
time = []
delta_Y = []
all_phases = []

for t, dy, phases in results:
    time.append(t)
    delta_Y.append(dy)
    all_phases.append(phases)
    ax2.scatter(np.array(phases)[:-1], np.array(phases)[1:]-np.array(phases)[:-1], s=1)
    ax2.set(
        xscale='log',
        xlabel=r'$\phi_{i}$',
        ylabel=r'$\Delta \phi$ per cycle',
    )
    ax2.axhline(0, c='k', alpha=0.5, linewidth=0.5)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots()
for phases in all_phases:
    phases = np.array(phases)
    ax.scatter(phases[:-1], (phases[1:]-phases[:-1]) + 0.1*2*np.pi, alpha=0.1, c='k')

ax.axhline()
ax.axhline(0.1*2*np.pi, ls='--', c='k', alpha=0.5)
    
ax.set(
    xlabel=r'$\phi_{i}$',
    ylabel=r'$\Delta \phi_{i+1}$ per cycle',
    xticks=np.arange(0,2*np.pi+1e-3,np.pi/2),
    xticklabels=[r'$0$',r'$\pi/2$',r'$\pi$',r'$3\pi/2$',r'$2\pi$'],
);